# 1. Extração do Grafo

Este notebook é o equivalente interativo do script `src/01_extracao_grafo.py`.
O objetivo desta etapa é ler o dataset gigante `quotes_2008-08.txt` e extrair as arestas (links de citação) que aconteceram nos dois primeiros dias do mês, salvando o resultado em um arquivo menor e muito mais fácil de manipular.


> **⚠️ AVISO IMPORTANTE**
>
> Para executar este Notebook específico, você precisa do dataset **bruto** do Memetracker, que possui cerca de 4 GB e por isso não foi salvo no repositório. Para isso basta ver o arquivo `AVISO_SOBRE_OS_DADOS.md` na pasta **data**.


In [ ]:
import os

input_file = "../data/quotes_2008-08.txt"
output_file = "../data/edges_2days.csv"
os.makedirs("../data", exist_ok=True)

# Vamos filtrar para os dias 01 e 02 de Agosto.
valid_days = ["2008-08-01", "2008-08-02"]
stop_day = "2008-08-03"

current_p = None
in_valid_time = False
edges_count = 0

print(f"Lendo {input_file} e extraindo arestas para {output_file}...")

Abaixo o loop de extração das arestas. Como o arquivo lido está ordenado por tempo, o laço `for` para a leitura instantaneamente ao encontrar a primeira entrada do dia 03, economizando gigabytes de leitura inútil na memória!

In [ ]:
with open(input_file, 'rt', encoding='utf-8', errors='ignore') as fin, \
     open(output_file, 'wt', encoding='utf-8') as fout:
    
    # Escrever cabeçalho
    fout.write("source,target\n")
    
    for line in fin:
        line = line.strip()
        if not line:
            continue
            
        prefix = line[0]
        
        if prefix == 'P':
            parts = line.split('\t')
            if len(parts) > 1:
                current_p = parts[1]
                in_valid_time = False # Reset para a nova postagem
                
        elif prefix == 'T':
            parts = line.split('\t')
            if len(parts) > 1:
                time_str = parts[1]
                
                if any(time_str.startswith(day) for day in valid_days):
                    in_valid_time = True
                elif time_str.startswith(stop_day):
                    # Como o arquivo é cronológico, podemos parar a leitura totalmente aqui e poupar tempo
                    print(f"Chegamos ao dia {stop_day}. Parando a extração.")
                    break
                else:
                    in_valid_time = False
                    
        elif prefix == 'L':
            if in_valid_time and current_p:
                parts = line.split('\t')
                if len(parts) > 1:
                    target_l = parts[1]
                    fout.write(f"{current_p},{target_l}\n")
                    edges_count += 1
                    
                    if edges_count % 100000 == 0:
                        print(f"Extraídas {edges_count} arestas...")

print(f"Extração concluída! Total de arestas encontradas nos 2 dias: {edges_count}")